# Protein Contact Map Prediction with CNNs

This notebook demonstrates how to build a deep learning model for protein contact prediction using 2D convolutional neural networks.

**Learning Objectives:**
- Understand the contact prediction problem
- Build pairwise feature representations
- Implement a 2D ResNet architecture
- Handle class imbalance in contact prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install torch numpy matplotlib seaborn scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, average_precision_score

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Understanding Contact Maps

A contact map is an L x L binary matrix where:
- L = protein length
- C[i,j] = 1 if residues i and j are in contact (distance < 8A)
- C[i,j] = 0 otherwise

In [ ]:
def generate_synthetic_protein(length, num_helices=2, num_sheets=1):
    """
    Generate a synthetic protein structure with secondary structure elements.
    Returns CA coordinates.
    """
    coords = np.zeros((length, 3))
    
    # Generate backbone along x-axis with some randomness
    for i in range(length):
        coords[i, 0] = i * 3.8  # CA-CA distance ~3.8A
        coords[i, 1] = np.random.randn() * 2
        coords[i, 2] = np.random.randn() * 2
    
    # Add helix-like contacts (i, i+4)
    helix_regions = np.random.choice(length - 10, num_helices, replace=False)
    for start in helix_regions:
        for i in range(start, min(start + 10, length - 4)):
            # Bring i and i+4 closer
            mid = (coords[i] + coords[i+4]) / 2
            coords[i] = mid + np.random.randn(3) * 1
            coords[i+4] = mid + np.random.randn(3) * 1
    
    # Add some long-range contacts (beta sheet-like)
    if length > 30 and num_sheets > 0:
        for _ in range(num_sheets):
            i = np.random.randint(5, length // 3)
            j = np.random.randint(2 * length // 3, length - 5)
            # Bring these regions together
            for di in range(-3, 4):
                if 0 <= i + di < length and 0 <= j - di < length:
                    mid = (coords[i + di] + coords[j - di]) / 2
                    coords[i + di] = mid + np.random.randn(3) * 2
                    coords[j - di] = mid + np.random.randn(3) * 2
    
    return coords

def compute_distance_matrix(coords):
    """Compute pairwise CA-CA distance matrix."""
    diff = coords[:, None, :] - coords[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))

def compute_contact_map(coords, threshold=8.0):
    """Compute binary contact map."""
    dist_matrix = compute_distance_matrix(coords)
    return (dist_matrix < threshold).astype(np.float32)

# Generate example protein
coords = generate_synthetic_protein(80)
dist_matrix = compute_distance_matrix(coords)
contact_map = compute_contact_map(coords)

print(f"Protein length: {len(coords)}")
print(f"Contact map shape: {contact_map.shape}")
print(f"Contact density: {contact_map.sum() / contact_map.size:.3f}")

In [ ]:
# Visualize distance and contact maps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distance matrix
im0 = axes[0].imshow(dist_matrix, cmap='viridis_r', origin='lower')
axes[0].set_title('Distance Matrix')
axes[0].set_xlabel('Residue j')
axes[0].set_ylabel('Residue i')
plt.colorbar(im0, ax=axes[0], label='Distance (A)')

# Contact map
axes[1].imshow(contact_map, cmap='Blues', origin='lower')
axes[1].set_title('Contact Map (< 8A)')
axes[1].set_xlabel('Residue j')
axes[1].set_ylabel('Residue i')

# Long-range contacts only
L = len(coords)
sep_mask = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :]) >= 12
long_range = contact_map * sep_mask
axes[2].imshow(long_range, cmap='Reds', origin='lower')
axes[2].set_title('Long-range Contacts (|i-j| >= 12)')
axes[2].set_xlabel('Residue j')
axes[2].set_ylabel('Residue i')

plt.tight_layout()
plt.show()

print(f"\nTotal contacts: {int(contact_map.sum())}")
print(f"Long-range contacts: {int(long_range.sum())}")

## Part 2: Creating Features for Contact Prediction

For contact prediction, we need:
1. **Sequence features** (per-residue): one-hot, PSSM, etc.
2. **Pairwise features**: sequence separation, coevolution signals

In [ ]:
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def one_hot_encode(sequence):
    """One-hot encode protein sequence."""
    encoding = np.zeros((len(sequence), 20), dtype=np.float32)
    for i, aa in enumerate(sequence):
        if aa in AA_TO_IDX:
            encoding[i, AA_TO_IDX[aa]] = 1.0
    return encoding

def generate_random_sequence(length):
    """Generate random protein sequence."""
    return ''.join(np.random.choice(list(AMINO_ACIDS), size=length))

def create_pairwise_features(L):
    """
    Create pairwise features for contact prediction.
    
    Returns:
        (L, L, num_features) array
    """
    # Sequence separation (normalized)
    i_idx = np.arange(L)[:, None]
    j_idx = np.arange(L)[None, :]
    sep = np.abs(i_idx - j_idx).astype(np.float32)
    sep_normalized = sep / L
    
    # One-hot encode separation ranges
    sep_short = (sep < 6).astype(np.float32)   # Local
    sep_medium = ((sep >= 6) & (sep < 12)).astype(np.float32)  # Medium
    sep_long = (sep >= 12).astype(np.float32)  # Long-range
    
    # Stack features
    features = np.stack([
        sep_normalized,
        sep_short,
        sep_medium,
        sep_long
    ], axis=-1)
    
    return features

# Test
seq = generate_random_sequence(80)
seq_features = one_hot_encode(seq)
pair_features = create_pairwise_features(80)

print(f"Sequence features shape: {seq_features.shape}")
print(f"Pairwise features shape: {pair_features.shape}")

In [ ]:
def outer_product(seq_features):
    """
    Create pairwise representation via outer product.
    
    Args:
        seq_features: (L, d) sequence features
    
    Returns:
        (L, L, 2d) pairwise features
    """
    L, d = seq_features.shape
    
    # Tile to create pairwise
    feat_i = np.tile(seq_features[:, None, :], (1, L, 1))  # (L, L, d)
    feat_j = np.tile(seq_features[None, :, :], (L, 1, 1))  # (L, L, d)
    
    # Concatenate
    pairwise = np.concatenate([feat_i, feat_j], axis=-1)  # (L, L, 2d)
    
    return pairwise

# Create full feature representation
outer_features = outer_product(seq_features)
all_pair_features = np.concatenate([outer_features, pair_features], axis=-1)

print(f"Outer product features: {outer_features.shape}")
print(f"All pairwise features: {all_pair_features.shape}")

## Part 3: Building the Contact Prediction Model

We'll use a 2D ResNet architecture that processes the LxL pairwise features.

In [ ]:
class ResBlock2D(nn.Module):
    """Residual block for 2D convolutions."""
    
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__()
        
        padding = dilation  # Same padding with dilation
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 
                               padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 
                               padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection
        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.skip = nn.Identity()
    
    def forward(self, x):
        residual = self.skip(x)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        
        return F.relu(x + residual)

In [ ]:
class ContactPredictor(nn.Module):
    """
    2D ResNet for contact map prediction.
    
    Architecture:
    1. Embed sequence features
    2. Create pairwise representation via outer product
    3. Process with 2D ResNet
    4. Predict contact probabilities
    """
    
    def __init__(self, seq_dim=20, pair_dim=4, hidden_dim=64, num_blocks=8):
        super().__init__()
        
        # Sequence embedding
        self.seq_embed = nn.Sequential(
            nn.Linear(seq_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )
        
        # Initial pairwise projection
        # Input: 2 * hidden_dim (outer product) + pair_dim (explicit features)
        pair_input_dim = 2 * hidden_dim + pair_dim
        self.pair_proj = nn.Conv2d(pair_input_dim, hidden_dim, 1)
        
        # ResNet blocks with varying dilation
        self.blocks = nn.ModuleList()
        dilations = [1, 2, 4, 8, 1, 2, 4, 8][:num_blocks]
        
        for dilation in dilations:
            self.blocks.append(ResBlock2D(hidden_dim, hidden_dim, dilation))
        
        # Output layer
        self.output = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim // 2, 1),
            nn.ReLU(),
            nn.Conv2d(hidden_dim // 2, 1, 1)
        )
    
    def forward(self, seq_features, pair_features):
        """
        Args:
            seq_features: (batch, L, seq_dim)
            pair_features: (batch, L, L, pair_dim)
        
        Returns:
            contact_logits: (batch, L, L)
        """
        batch_size, L, _ = seq_features.shape
        
        # Embed sequence
        seq = self.seq_embed(seq_features)  # (batch, L, hidden)
        
        # Outer product
        seq_i = seq.unsqueeze(2).expand(-1, -1, L, -1)  # (batch, L, L, hidden)
        seq_j = seq.unsqueeze(1).expand(-1, L, -1, -1)  # (batch, L, L, hidden)
        pair = torch.cat([seq_i, seq_j, pair_features], dim=-1)  # (batch, L, L, 2h+p)
        
        # Reshape for conv2d: (batch, channels, L, L)
        pair = pair.permute(0, 3, 1, 2)
        
        # Initial projection
        pair = self.pair_proj(pair)
        
        # ResNet blocks
        for block in self.blocks:
            pair = block(pair)
        
        # Output
        logits = self.output(pair).squeeze(1)  # (batch, L, L)
        
        # Symmetrize
        logits = (logits + logits.transpose(1, 2)) / 2
        
        return logits

In [ ]:
# Test the model
model = ContactPredictor(seq_dim=20, pair_dim=4, hidden_dim=64, num_blocks=8)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Test forward pass
test_seq = torch.randn(2, 50, 20).to(device)
test_pair = torch.randn(2, 50, 50, 4).to(device)

with torch.no_grad():
    output = model(test_seq, test_pair)
    print(f"\nInput shapes: seq={test_seq.shape}, pair={test_pair.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")

## Part 4: Dataset and Training

In [ ]:
class ContactDataset(Dataset):
    """Dataset for contact prediction."""
    
    def __init__(self, num_samples=500, min_len=50, max_len=100):
        self.samples = []
        
        for _ in range(num_samples):
            # Random length
            length = np.random.randint(min_len, max_len + 1)
            
            # Generate structure and contact map
            coords = generate_synthetic_protein(length)
            contacts = compute_contact_map(coords)
            
            # Generate sequence and features
            sequence = generate_random_sequence(length)
            seq_features = one_hot_encode(sequence)
            pair_features = create_pairwise_features(length)
            
            self.samples.append({
                'seq_features': seq_features,
                'pair_features': pair_features,
                'contacts': contacts,
                'length': length
            })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            'seq_features': torch.tensor(sample['seq_features'], dtype=torch.float),
            'pair_features': torch.tensor(sample['pair_features'], dtype=torch.float),
            'contacts': torch.tensor(sample['contacts'], dtype=torch.float),
            'length': sample['length']
        }

def collate_fn(batch):
    """Collate variable-length proteins."""
    max_len = max(b['length'] for b in batch)
    batch_size = len(batch)
    
    seq_features = torch.zeros(batch_size, max_len, 20)
    pair_features = torch.zeros(batch_size, max_len, max_len, 4)
    contacts = torch.zeros(batch_size, max_len, max_len)
    masks = torch.zeros(batch_size, max_len, max_len)
    
    for i, b in enumerate(batch):
        L = b['length']
        seq_features[i, :L] = b['seq_features']
        pair_features[i, :L, :L] = b['pair_features']
        contacts[i, :L, :L] = b['contacts']
        masks[i, :L, :L] = 1.0
    
    return {
        'seq_features': seq_features,
        'pair_features': pair_features,
        'contacts': contacts,
        'masks': masks
    }

# Create datasets
print("Creating datasets...")
train_dataset = ContactDataset(num_samples=400, min_len=50, max_len=80)
val_dataset = ContactDataset(num_samples=100, min_len=50, max_len=80)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
def focal_loss(pred, target, mask, gamma=2.0, alpha=0.25):
    """
    Focal loss for imbalanced contact prediction.
    
    Args:
        pred: (batch, L, L) logits
        target: (batch, L, L) binary targets
        mask: (batch, L, L) validity mask
        gamma: focusing parameter
        alpha: weight for positive class
    """
    # Binary cross entropy
    bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
    
    # Probability
    p = torch.sigmoid(pred)
    p_t = target * p + (1 - target) * (1 - p)
    
    # Focal weight
    focal_weight = (1 - p_t) ** gamma
    
    # Alpha weight
    alpha_weight = target * alpha + (1 - target) * (1 - alpha)
    
    # Weighted loss
    loss = focal_weight * alpha_weight * bce
    
    # Apply mask
    loss = (loss * mask).sum() / mask.sum().clamp(min=1)
    
    return loss

In [ ]:
def precision_at_l(pred, target, mask, k=1, min_sep=6):
    """
    Compute precision at L/k for long-range contacts.
    
    Args:
        pred: (L, L) predictions
        target: (L, L) ground truth
        mask: (L, L) validity mask
        k: denominator (L/k predictions)
        min_sep: minimum sequence separation for long-range
    """
    L = pred.shape[0]
    n_contacts = max(L // k, 1)
    
    # Create separation mask
    i_idx = torch.arange(L, device=pred.device)[:, None]
    j_idx = torch.arange(L, device=pred.device)[None, :]
    sep_mask = (torch.abs(i_idx - j_idx) >= min_sep).float()
    
    # Upper triangular only (symmetric matrix)
    upper_mask = (i_idx < j_idx).float()
    
    # Combined mask
    valid_mask = mask * sep_mask * upper_mask
    
    # Mask invalid positions
    pred_masked = pred.clone()
    pred_masked[valid_mask == 0] = -float('inf')
    
    # Get top-k predictions
    flat_pred = pred_masked.flatten()
    _, top_idx = flat_pred.topk(min(n_contacts, int(valid_mask.sum().item())))
    
    # Check against ground truth
    flat_target = target.flatten()
    correct = flat_target[top_idx].sum().item()
    precision = correct / n_contacts
    
    return precision

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch in loader:
        seq = batch['seq_features'].to(device)
        pair = batch['pair_features'].to(device)
        contacts = batch['contacts'].to(device)
        masks = batch['masks'].to(device)
        
        # Forward
        logits = model(seq, pair)
        loss = focal_loss(logits, contacts, masks)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    precisions = {1: [], 2: [], 5: []}
    
    for batch in loader:
        seq = batch['seq_features'].to(device)
        pair = batch['pair_features'].to(device)
        contacts = batch['contacts'].to(device)
        masks = batch['masks'].to(device)
        
        logits = model(seq, pair)
        loss = focal_loss(logits, contacts, masks)
        total_loss += loss.item()
        
        # Compute precision for each sample
        probs = torch.sigmoid(logits)
        for i in range(len(logits)):
            for k in [1, 2, 5]:
                p = precision_at_l(probs[i], contacts[i], masks[i], k=k)
                precisions[k].append(p)
    
    avg_precisions = {k: np.mean(v) for k, v in precisions.items()}
    return total_loss / len(loader), avg_precisions

In [ ]:
# Initialize model and training
model = ContactPredictor(seq_dim=20, pair_dim=4, hidden_dim=64, num_blocks=8)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

# Training loop
n_epochs = 30
history = {'train_loss': [], 'val_loss': [], 'P@L': [], 'P@L/2': [], 'P@L/5': []}

print("Training...")
for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_loss, precisions = evaluate(model, val_loader, device)
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['P@L'].append(precisions[1])
    history['P@L/2'].append(precisions[2])
    history['P@L/5'].append(precisions[5])
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"P@L: {precisions[1]:.3f} | P@L/2: {precisions[2]:.3f} | P@L/5: {precisions[5]:.3f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Focal Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision
axes[1].plot(history['P@L'], label='P@L')
axes[1].plot(history['P@L/2'], label='P@L/2')
axes[1].plot(history['P@L/5'], label='P@L/5')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision at L/k (Long-range)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Visualizing Predictions

In [ ]:
# Get a sample prediction
model.eval()
sample = val_dataset[0]

with torch.no_grad():
    seq = sample['seq_features'].unsqueeze(0).to(device)
    pair = sample['pair_features'].unsqueeze(0).to(device)
    
    logits = model(seq, pair)
    pred = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    target = sample['contacts'].numpy()

L = sample['length']
pred = pred[:L, :L]
target = target[:L, :L]

print(f"Protein length: {L}")

In [ ]:
# Visualize prediction vs ground truth
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Ground truth
axes[0].imshow(target, cmap='Blues', origin='lower', vmin=0, vmax=1)
axes[0].set_title('Ground Truth Contacts')
axes[0].set_xlabel('Residue j')
axes[0].set_ylabel('Residue i')

# Predicted probabilities
im = axes[1].imshow(pred, cmap='Reds', origin='lower', vmin=0, vmax=1)
axes[1].set_title('Predicted Probabilities')
axes[1].set_xlabel('Residue j')
plt.colorbar(im, ax=axes[1], label='P(contact)')

# Overlay (TP=green, FP=red, FN=blue)
binary_pred = (pred > 0.5).astype(float)
overlay = np.zeros((L, L, 3))
tp = (binary_pred == 1) & (target == 1)
fp = (binary_pred == 1) & (target == 0)
fn = (binary_pred == 0) & (target == 1)

overlay[tp, 1] = 1  # Green: True Positive
overlay[fp, 0] = 1  # Red: False Positive
overlay[fn, 2] = 1  # Blue: False Negative

axes[2].imshow(overlay, origin='lower')
axes[2].set_title('Prediction Analysis')
axes[2].set_xlabel('Residue j')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='True Positive'),
    Patch(facecolor='red', label='False Positive'),
    Patch(facecolor='blue', label='False Negative')
]
axes[2].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nStatistics (threshold=0.5):")
print(f"  True Positives: {tp.sum()}")
print(f"  False Positives: {fp.sum()}")
print(f"  False Negatives: {fn.sum()}")
print(f"  Precision: {tp.sum() / (tp.sum() + fp.sum()):.3f}" if tp.sum() + fp.sum() > 0 else "  Precision: N/A")
print(f"  Recall: {tp.sum() / (tp.sum() + fn.sum()):.3f}" if tp.sum() + fn.sum() > 0 else "  Recall: N/A")

In [ ]:
# Precision-Recall curve
# Focus on long-range contacts
sep_mask = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :]) >= 12
upper_mask = np.triu(np.ones((L, L)), k=1).astype(bool)
valid_mask = sep_mask & upper_mask

y_true = target[valid_mask]
y_pred = pred[valid_mask]

precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
ap = average_precision_score(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, 'b-', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (Long-range Contacts)\nAP = {ap:.3f}')
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.show()

## Summary

In this notebook, we learned:

1. **Contact maps** represent pairwise residue contacts in proteins
2. **Pairwise features** can be created via outer product of sequence features
3. **2D ResNets** with dilated convolutions are effective for contact prediction
4. **Focal loss** helps handle severe class imbalance (few contacts, many non-contacts)
5. **Precision at L/k** is the standard evaluation metric for contact prediction

**Key takeaways:**
- Contact prediction is fundamentally a pairwise classification problem
- Class imbalance is a major challenge (~2-5% positive rate)
- Long-range contacts (|i-j| >= 12) are most informative for structure
- Real-world models use coevolution features (MSA-derived) which greatly improve accuracy